# Sports Platform Coverage Gap Analysis

**Author:** Alessandro Attene  
**Type:** Exploratory Data Analysis (EDA)  

---

## Business Questions

A sports management platform operating in Italy wants to understand where its biggest growth opportunities lie. This analysis answers five key questions:

1. **Where should the platform expand first?** Which provinces show the largest gap between registered sports entities and platform coverage?
2. **Which sports represent the biggest untapped opportunity?** What is the sport mix on the platform, and which categories are under-represented?
3. **How is the platform growing over time?** What does the registration trajectory look like?
4. **What should a phased expansion strategy look like?** How can provinces be prioritized based on market size and current gap?
5. **What is the total addressable market not yet reached?** How large is the coverage gap in absolute terms?

In [ ]:
import sys
import json
import warnings
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.config import (
    REGISTRY_COUNTS_CSV,
    PLATFORM_COUNTS_CSV,
    PLATFORM_RAW_DIR,
    REGION_CODE_TO_NAME,
    ANALYSIS_DIR,
)

warnings.filterwarnings("ignore", category=FutureWarning)

# Consistent style
sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = sns.color_palette("tab10")
COLOR_PRIMARY = "#2563EB"
COLOR_SECONDARY = "#F59E0B"
COLOR_GAP = "#DC2626"
COLOR_COVERED = "#16A34A"

FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir:  {FIGURES_DIR}")

### Data Sources

| Source | Description | Granularity |
|--------|-------------|-------------|
| **Registry** | Official sports registry — total registered entities by province | Province (~107) |
| **Platform** | Sports management platform — entities currently on the platform | Province + Sport + Year |

> **Privacy note:** Platform raw data has been sanitized at collection time. Only `sport`, `registration_year`, `province_abbr`, and `region_code` are retained. No personal data (names, addresses, coordinates) is stored.

---
## 1. Data Loading & Preparation

In [ ]:
# Guard: verify that required data files exist before loading.
# On a fresh clone, data/ is gitignored. Populate it by running the pipeline:
#   FETCH_PLATFORM_DATA=true FETCH_REGISTRY_DATA=true python -m run_pipeline
_required = [REGISTRY_COUNTS_CSV, PLATFORM_COUNTS_CSV]
_missing = [p for p in _required if not p.exists()]
if _missing:
    raise FileNotFoundError(
        "Required data files not found. Populate data/ before running this notebook.\n"
        "  Run: FETCH_PLATFORM_DATA=true FETCH_REGISTRY_DATA=true python -m run_pipeline\n\n"
        "Missing:\n" + "\n".join(f"  - {p}" for p in _missing)
    )
print("Data files found. Ready to load.")

In [ ]:
# Registry data: entity counts by province
df_registry = pd.read_csv(REGISTRY_COUNTS_CSV)
print(f"Registry: {df_registry.shape[0]} provinces, {df_registry['entities_total'].sum():,} entities")
df_registry.head()

In [ ]:
# Platform data: entity counts by province
df_platform = pd.read_csv(PLATFORM_COUNTS_CSV)
print(f"Platform: {df_platform.shape[0]} provinces, {df_platform['platform_entities'].sum():,} entities")
df_platform.head()

In [ ]:
# Load raw platform entities for sport-level and temporal analysis
raw_path = PLATFORM_RAW_DIR / "platform_entities.json"

if raw_path.exists():
    with open(raw_path, encoding="utf-8") as f:
        raw_data = json.load(f)

    df_entities = pd.DataFrame(raw_data["items"])
    print(f"Raw platform entities: {len(df_entities):,}")
    print(f"Multi-sport entities: {df_entities['sport'].apply(len).gt(1).sum():,} ({df_entities['sport'].apply(len).gt(1).mean():.0%})")

    # Explode sport list: one row per (entity, sport) pair
    df_exploded = df_entities.explode("sport").reset_index(drop=True)
    print(f"Sport-entity pairs after explode: {len(df_exploded):,}")
    print(f"Unique sports: {df_exploded['sport'].nunique()}")
else:
    print(f"Raw platform payload not found at {raw_path}.")
    print(
        "Sport-level and temporal analyses will be skipped.\n"
        "To generate this file:\n"
        "  FETCH_PLATFORM_DATA=true python -m run_pipeline"
    )
    # Empty DataFrames so downstream cells do not raise NameError
    df_entities = pd.DataFrame(columns=["sport", "registration_year", "province_abbr", "region_code"])
    df_exploded = pd.DataFrame(columns=["sport", "registration_year", "province_abbr", "region_code", "region_name", "sport_macro"])

In [ ]:
# Sport macro-category mapping (174 raw sports -> 12 categories)
FITNESS_WELLNESS = "Fitness & Wellness"
WINTER_MOUNTAIN = "Winter & Mountain"
TENNIS_RACQUET = "Tennis & Racquet"
SWIMMING_WATER = "Swimming & Water"
GYMNASTICS_DANCE = "Gymnastics & Dance"
MARTIAL_ARTS = "Martial Arts"
SPORT_ENTITY = "Sport & Entity"

SPORT_MACRO: dict[str, str] = {
    # Football
    "football": "Football", "futsal": "Football", "beach-soccer": "Football",
    "seven_a_side_football": "Football", "calcio-camminato": "Football",
    "american_football": "Football",
    # Basketball
    "basketball": "Basketball", "baskin": "Basketball",
    # Volleyball
    "volleyball": "Volleyball", "beach_volley": "Volleyball",
    # Tennis & Racquet
    "tennis": TENNIS_RACQUET, "padel": TENNIS_RACQUET,
    "beach_tennis": TENNIS_RACQUET, "table_tennis": TENNIS_RACQUET,
    "pickleball": TENNIS_RACQUET, "squash": TENNIS_RACQUET,
    "badminton": TENNIS_RACQUET, "padbol": TENNIS_RACQUET,
    # Swimming & Water
    "swimming": SWIMMING_WATER, "synchronized_swimming": SWIMMING_WATER,
    "waterpolo": SWIMMING_WATER, "diving": SWIMMING_WATER,
    "apnea": SWIMMING_WATER, "apnea-indoor": SWIMMING_WATER,
    "nuoto-pinnato": SWIMMING_WATER, "freediving": SWIMMING_WATER,
    "attivita-subacquee": SWIMMING_WATER,
    "sailing": SWIMMING_WATER, "rowing": SWIMMING_WATER,
    "canoeing": SWIMMING_WATER, "kayak": SWIMMING_WATER,
    "dragon-boat": SWIMMING_WATER, "surf": SWIMMING_WATER,
    "windsurf": SWIMMING_WATER, "wakeboard": SWIMMING_WATER,
    "sup": SWIMMING_WATER, "waszp": SWIMMING_WATER,
    # Cycling
    "cycling": "Cycling", "mountain_bike": "Cycling",
    "bmx": "Cycling", "bmx-freestyle": "Cycling",
    # Athletics & Running
    "athletics": "Athletics", "light_athletics": "Athletics",
    "trail_running": "Athletics", "triathlon": "Athletics",
    "running_skating": "Athletics",
    # Gymnastics & Dance
    "gymnastics": GYMNASTICS_DANCE, "artistic_gymnastics": GYMNASTICS_DANCE,
    "rhythmic_gymnastics": GYMNASTICS_DANCE, "acrobatic_gymnastics": GYMNASTICS_DANCE,
    "functional_gymnastics": GYMNASTICS_DANCE, "postural_gymnastics": GYMNASTICS_DANCE,
    "soft_gymnastics": GYMNASTICS_DANCE, "gymnastics_on_the_apparatus": GYMNASTICS_DANCE,
    "cheerleading": GYMNASTICS_DANCE, "cheerdance": GYMNASTICS_DANCE,
    "cheerability": GYMNASTICS_DANCE, "teamgym": GYMNASTICS_DANCE,
    "trampoline": GYMNASTICS_DANCE, "twirling": GYMNASTICS_DANCE,
    "majorettes": GYMNASTICS_DANCE, "psychomotricity": GYMNASTICS_DANCE,
    "dancing": GYMNASTICS_DANCE, "sport_dance": GYMNASTICS_DANCE,
    # Martial Arts
    "martial_arts": MARTIAL_ARTS, "judo": MARTIAL_ARTS, "karate": MARTIAL_ARTS,
    "taekwondo": MARTIAL_ARTS, "jujitsu": MARTIAL_ARTS, "aikido": MARTIAL_ARTS,
    "kung_fu": MARTIAL_ARTS, "mma": MARTIAL_ARTS, "kick_boxing": MARTIAL_ARTS,
    "boxing": MARTIAL_ARTS, "boxing_fitness": MARTIAL_ARTS,
    "brazilian_jiu_jitsu": MARTIAL_ARTS, "capoeira": "Martial Arts",
    "grappling": MARTIAL_ARTS, "kendo": MARTIAL_ARTS, "iaido": MARTIAL_ARTS,
    "jodo": MARTIAL_ARTS, "naginata": MARTIAL_ARTS, "savate": MARTIAL_ARTS,
    "personal_defense": MARTIAL_ARTS, "vovinam": MARTIAL_ARTS,
    "wing_chun": MARTIAL_ARTS, "wushu": MARTIAL_ARTS, "wrestling": MARTIAL_ARTS,
    "qwan-ki-do": MARTIAL_ARTS, "fencing": MARTIAL_ARTS,
    "tai_chi_chuan": MARTIAL_ARTS,
    # Fitness & Wellness
    "fitness": FITNESS_WELLNESS, "pilates": FITNESS_WELLNESS,
    "yoga": FITNESS_WELLNESS, "calistenics": FITNESS_WELLNESS,
    "crosstraining": FITNESS_WELLNESS, "step_and_tone": FITNESS_WELLNESS,
    "bodybuilding": FITNESS_WELLNESS, "culturismo": FITNESS_WELLNESS,
    "powerlifting": FITNESS_WELLNESS, "kettlebell": FITNESS_WELLNESS,
    "heavy_athletics": FITNESS_WELLNESS, "weightlifting": FITNESS_WELLNESS,
    "nordic_walking": FITNESS_WELLNESS,
    # Winter & Mountain
    "alpine-downhill-skiing": WINTER_MOUNTAIN, "cross_country_skiing": WINTER_MOUNTAIN,
    "freestyle-skiing": WINTER_MOUNTAIN, "skiing": WINTER_MOUNTAIN,
    "snowboard": WINTER_MOUNTAIN, "curling": WINTER_MOUNTAIN,
    "ice_skating": WINTER_MOUNTAIN, "ski-mountaineering": WINTER_MOUNTAIN,
    "speed\u200b-skiing": WINTER_MOUNTAIN,
    "climbing": WINTER_MOUNTAIN, "sport_climbing": WINTER_MOUNTAIN,
    "alpinism": WINTER_MOUNTAIN, "canyoning": WINTER_MOUNTAIN,
    "trekking": WINTER_MOUNTAIN, "parkour": WINTER_MOUNTAIN,
}

df_exploded["sport_macro"] = df_exploded["sport"].map(SPORT_MACRO).fillna("Other Sports")

print(f"Macro categories: {df_exploded['sport_macro'].nunique()}")
print("\nSport-entity pairs by macro category:")
print(df_exploded["sport_macro"].value_counts().to_string())

In [ ]:
# Fix known data quality issue: region_code 'VAO' -> 'VDA' (Valle d'Aosta)
REGION_CODE_FIX: dict[str, str] = {"VAO": "VDA"}
REGION_NAME_MAP: dict[str, str] = {
    **REGION_CODE_TO_NAME,
    "VAO": "Valle d'Aosta",
}

# Apply fix to platform data
df_platform["region_code"] = df_platform["region_code"].replace(REGION_CODE_FIX)
df_platform["region_name"] = df_platform["region_code"].map(REGION_CODE_TO_NAME).fillna(df_platform["region_name"])

# Apply fix to exploded entities
df_exploded["region_code"] = df_exploded["region_code"].replace(REGION_CODE_FIX)
df_exploded["region_name"] = df_exploded["region_code"].map(REGION_NAME_MAP)

# Merge registry + platform at province level
df = df_registry.merge(
    df_platform[["province_abbr", "platform_entities"]],
    on="province_abbr",
    how="left",
)
df["platform_entities"] = df["platform_entities"].fillna(0).astype(int)
df["coverage_gap"] = df["entities_total"] - df["platform_entities"]
df["coverage_ratio"] = df["platform_entities"] / df["entities_total"]

registry_provinces = len(df_registry)
total_provinces = 107

print(f"Merged dataset: {len(df)} provinces (registry coverage: {registry_provinces}/{total_provinces})")
print(f"Total registry entities:  {df['entities_total'].sum():,}")
print(f"Total platform entities:  {df['platform_entities'].sum():,}")
print(f"Total coverage gap:       {df['coverage_gap'].sum():,}")

if registry_provinces < total_provinces:
    print(f"\n⚠ Registry data covers {registry_provinces} of {total_provinces} provinces.")
    print("  Run the full pipeline (FETCH_REGISTRY_DATA=true) for complete gap analysis.")
    print("  Platform-only analyses (sport, temporal, geographic distribution) are unaffected.")

---
## 2. Data Quality Overview

In [ ]:
print("=== Registry ===")
display(df_registry.describe())
print(f"Null values: {df_registry.isna().sum().sum()}")

print("\n=== Platform (province-level) ===")
display(df_platform.describe())
print(f"Null values: {df_platform.isna().sum().sum()}")

print("\n=== Platform (entity-level) ===")
total_entities = len(df_entities)
if total_entities > 0:
    with_year = df_entities["registration_year"].notna().sum()
    print(f"Entities: {total_entities:,}")
    print(f"With registration_year: {with_year:,} ({with_year/total_entities:.0%})")
    print(f"Without registration_year: {total_entities - with_year:,}")
    print(f"Year range: {df_entities['registration_year'].min():.0f} - {df_entities['registration_year'].max():.0f}")
    print(f"Unique provinces: {df_entities['province_abbr'].nunique()}")
    print(f"Unique regions: {df_entities['region_code'].nunique()}")
else:
    print("No entity-level data available (raw platform payload not found).")

> **Note on sport-entity pairs:** 33% of entities offer multiple sports. When analyzing by sport, each entity is counted once per sport it offers. Sport-level totals therefore represent _sport-entity pairs_, not unique entities.

---
## 3. Platform Geographic Distribution

Before analyzing the coverage gap (which requires registry data), let's understand where the platform currently operates across Italy's 107 provinces.

In [ ]:
# Platform entities by region
df_plat_region = (
    df_platform
    .groupby("region_name", as_index=False)
    .agg(platform_total=("platform_entities", "sum"), province_count=("province_abbr", "count"))
    .sort_values("platform_total", ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(df_plat_region["region_name"], df_plat_region["platform_total"], color=COLOR_PRIMARY)

for bar, val in zip(bars, df_plat_region["platform_total"]):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)

ax.set_xlabel("Platform Entities")
ax.set_title("Platform Presence by Region", fontsize=14, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig(FIGURES_DIR / "platform_distribution_by_region.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Distribution of platform entities per province
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df_platform["platform_entities"], bins=30, color=COLOR_PRIMARY, edgecolor="white")
axes[0].set_xlabel("Entities per Province")
axes[0].set_ylabel("Number of Provinces")
axes[0].set_title("Distribution of Platform Entities per Province")
axes[0].axvline(df_platform["platform_entities"].median(), color=COLOR_GAP, linestyle="--",
                label=f"Median: {df_platform['platform_entities'].median():.0f}")
axes[0].legend()

# Box plot by region (top 10 regions)
top_regions = df_plat_region.nlargest(10, "platform_total")["region_name"].tolist()
df_top = df_platform[df_platform["region_name"].isin(top_regions)]
region_order = (
    df_top.groupby("region_name")["platform_entities"]
    .median().sort_values(ascending=False).index.tolist()
)
sns.boxplot(data=df_top, y="region_name", x="platform_entities", order=region_order,
            ax=axes[1], color=COLOR_PRIMARY)
axes[1].set_xlabel("Entities per Province")
axes[1].set_ylabel("")
axes[1].set_title("Province-Level Spread (Top 10 Regions)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "platform_province_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Key finding:** In the current dataset extract, the platform is present across the provinces included in the data snapshot, but concentration is highly uneven. Lombardia alone accounts for a disproportionate share of total entities, and within each region the distribution is heavily skewed toward the regional capital.

---
## 4. Coverage Gap Analysis

Comparing registry data (total addressable market) with platform presence to quantify the gap.

| KPI | Formula | Interpretation |
|-----|---------|----------------|
| **Coverage Ratio** | `platform_entities / entities_total` | 0 = no coverage, 1 = full coverage |
| **Coverage Gap** | `entities_total - platform_entities` | Absolute number of unreached entities |

In [ ]:
# Coverage Gap by Region (stacked: covered + gap)
df_region = (
    df.groupby("region_name", as_index=False)
    .agg(
        registry_total=("entities_total", "sum"),
        platform_total=("platform_entities", "sum"),
    )
)
df_region["coverage_gap"] = df_region["registry_total"] - df_region["platform_total"]
df_region["coverage_ratio"] = df_region["platform_total"] / df_region["registry_total"]
df_region = df_region.sort_values("registry_total", ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))

ax.barh(df_region["region_name"], df_region["platform_total"],
        color=COLOR_COVERED, label="Platform covered")
ax.barh(df_region["region_name"], df_region["coverage_gap"],
        left=df_region["platform_total"], color=COLOR_GAP, alpha=0.7, label="Coverage gap")

# Annotate with ratio text on the right
for idx, (_, row) in enumerate(df_region.iterrows()):
    ax.text(row["registry_total"] + 30, idx,
            f"{row['coverage_ratio']:.0%}", va="center", fontsize=8, color="gray")

ax.set_xlabel("Number of Sports Entities")
ax.set_title("Coverage Gap by Region: Platform vs Total Market", fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.savefig(FIGURES_DIR / "coverage_gap_by_region_stacked.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Scatter: Registry entities vs Platform entities per province
fig, ax = plt.subplots(figsize=(10, 8))

max_val = max(df["entities_total"].max(), df["platform_entities"].max()) * 1.1
ax.plot([0, max_val], [0, max_val], "--", color="gray", alpha=0.5, label="100% coverage line")

scatter = ax.scatter(
    df["entities_total"], df["platform_entities"],
    c=df["coverage_ratio"], cmap="RdYlGn", s=80, edgecolors="white", linewidth=0.5,
    vmin=0, vmax=0.5, zorder=5,
)
plt.colorbar(scatter, ax=ax, label="Coverage Ratio", shrink=0.8)

# Annotate top-5 by gap
top5 = df.nlargest(5, "coverage_gap")
for _, row in top5.iterrows():
    ax.annotate(
        f"{row['province_name']} ({row['province_abbr']})",
        (row["entities_total"], row["platform_entities"]),
        textcoords="offset points", xytext=(8, 8), fontsize=8,
        arrowprops={"arrowstyle":"->", "color":"gray", "lw":0.8},
    )

ax.set_xlabel("Registry Entities (Total Market)")
ax.set_ylabel("Platform Entities (Current Coverage)")
ax.set_title("Registry vs Platform: Province-Level Coverage", fontsize=14, fontweight="bold")
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "scatter_registry_vs_platform.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Coverage ratio distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of coverage ratio
axes[0].hist(df["coverage_ratio"], bins=20, color=COLOR_PRIMARY, edgecolor="white")
axes[0].set_xlabel("Coverage Ratio")
axes[0].set_ylabel("Number of Provinces")
axes[0].set_title("Distribution of Coverage Ratio")
axes[0].axvline(df["coverage_ratio"].median(), color=COLOR_GAP, linestyle="--",
                label=f"Median: {df['coverage_ratio'].median():.2%}")
axes[0].legend()

# Top 15 provinces by coverage gap
df_top_gap = df.nlargest(15, "coverage_gap")
ax2 = axes[1]
bars = ax2.barh(
    df_top_gap["province_name"] + " (" + df_top_gap["province_abbr"] + ")",
    df_top_gap["coverage_gap"],
    color=COLOR_GAP, alpha=0.8,
)
ax2.set_xlabel("Coverage Gap (entities)")
ax2.set_title("Top 15 Provinces by Coverage Gap")
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "coverage_ratio_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

**Key findings:**
- The vast majority of provinces show a **very low coverage ratio**, indicating the platform has reached only a small fraction of the total addressable market.
- The largest absolute gaps are concentrated in the most populated provinces.
- The scatter plot confirms that points cluster near the x-axis, far below the 100% coverage diagonal, highlighting a market largely untapped.

---
## 5. Sport-Level Analysis

The platform covers 174 individual sports. To identify strategic patterns, sports are grouped into 12 macro-categories.

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping top sports chart.")
else:
    # Top 15 sports (raw) on platform
    SPORT_ENTITY_PAIRS = "Sport-Entity Pairs"
    sport_counts = df_exploded["sport"].value_counts().head(15)

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(sport_counts.index[::-1], sport_counts.values[::-1], color=COLOR_PRIMARY)

    for bar, val in zip(bars, sport_counts.values[::-1]):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=9)

    ax.set_xlabel(SPORT_ENTITY_PAIRS)
    ax.set_title("Top 15 Sports on the Platform", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "top_sports_platform.png", dpi=150, bbox_inches="tight")
    plt.show()

SPORT_ENTITY_PAIRS = "Sport-Entity Pairs"

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport mix distribution chart.")
else:
    # Sport macro-category distribution
    macro_counts = df_exploded["sport_macro"].value_counts()
    macro_pct = (macro_counts / macro_counts.sum() * 100).round(1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Bar chart
    colors = sns.color_palette("tab10", n_colors=len(macro_counts))
    bars = axes[0].barh(macro_counts.index[::-1], macro_counts.values[::-1], color=colors[::-1])
    for bar, val, pct in zip(bars, macro_counts.values[::-1], macro_pct.values[::-1]):
        axes[0].text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                     f"{val:,} ({pct}%)", va="center", fontsize=9)
    axes[0].set_xlabel(SPORT_ENTITY_PAIRS)
    axes[0].set_title("Sport Mix by Macro-Category", fontsize=13, fontweight="bold")

    # Pie chart for top categories
    top_n = 8
    top_macro = macro_counts.head(top_n)
    other = pd.Series({"Other": macro_counts.iloc[top_n:].sum()})
    pie_data = pd.concat([top_macro, other])
    pie_colors = colors[:top_n] + [(0.8, 0.8, 0.8)]
    axes[1].pie(pie_data.values, labels=pie_data.index, autopct="%1.1f%%",
                colors=pie_colors, startangle=90, textprops={"fontsize": 9})
    axes[1].set_title("Sport Concentration", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "sport_mix_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    top3_share = macro_pct.iloc[:3].sum()
    print(f"Top 3 macro-categories account for {top3_share:.1f}% of all sport-entity pairs.")

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport × region heatmap.")
else:
    # Sport x Region heatmap (macro-categories)
    pivot_sport = df_exploded.pivot_table(
        index="region_name", columns="sport_macro", values="sport",
        aggfunc="count", fill_value=0,
    )

    # Sort regions by total, keep top macro-categories
    pivot_sport = pivot_sport.loc[
        pivot_sport.sum(axis=1).sort_values(ascending=False).index
    ]

    fig, ax = plt.subplots(figsize=(16, 9))
    sns.heatmap(
        pivot_sport, cmap="YlOrRd", linewidths=0.5, ax=ax,
        fmt=",d", annot=True, annot_kws={"size": 8},
        cbar_kws={"label": SPORT_ENTITY_PAIRS},
    )
    ax.set_title("Platform Presence: Sport Category x Region", fontsize=14, fontweight="bold")
    ax.set_ylabel("")
    ax.set_xlabel("")
    plt.xticks(rotation=35, ha="right")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "sport_region_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport diversity index.")
else:
    # Sport diversity index: how many distinct macro-categories per region
    diversity = (
        df_exploded.groupby("region_name")["sport_macro"]
        .nunique()
        .sort_values(ascending=False)
        .rename("sport_categories")
    )

    print("Sport Diversity Index (macro-categories per region):")
    print(f"  Max: {diversity.max()} categories ({diversity.idxmax()})")
    print(f"  Min: {diversity.min()} categories ({diversity.idxmin()})")
    print(f"  Mean: {diversity.mean():.1f}")
    print()
    display(diversity.to_frame().T)

**Key findings:**
- **Football dominates** the platform, representing the largest share of sport-entity pairs. Together with the next two categories, they account for over half of all activity.
- The heatmap reveals a **strong geographic concentration**: Lombardia, Lazio, and Veneto lead across almost every sport category.
- Sport diversity is relatively uniform across regions — most regions cover 10+ macro-categories — but the _depth_ (number of entities per category) varies dramatically.
- **Under-represented categories** like Athletics, Winter & Mountain, and Swimming & Water could represent niche expansion opportunities in regions with relevant infrastructure.

---
## 6. Growth Trajectory

Analyzing registration year data to understand the platform's growth pattern.

In [ ]:
if len(df_entities) > 0:
    # Registration year trend
    df_year = df_entities.dropna(subset=["registration_year"]).copy()
    df_year["registration_year"] = df_year["registration_year"].astype(int)

    year_counts = df_year["registration_year"].value_counts().sort_index()
    cumulative = year_counts.cumsum()

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Bar chart: new registrations per year
    bars = ax1.bar(year_counts.index.astype(str), year_counts.values, color=COLOR_PRIMARY,
                   label="New registrations", zorder=3)
    for bar, val in zip(bars, year_counts.values):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 15,
                 f"{val:,}", ha="center", fontsize=10, fontweight="bold")

    ax1.set_xlabel("Registration Year")
    ax1.set_ylabel("New Registrations", color=COLOR_PRIMARY)
    ax1.tick_params(axis="y", labelcolor=COLOR_PRIMARY)

    # Cumulative line on secondary axis
    ax2 = ax1.twinx()
    ax2.plot(year_counts.index.astype(str), cumulative.values, color=COLOR_SECONDARY,
             marker="o", linewidth=2.5, label="Cumulative", zorder=4)
    ax2.set_ylabel("Cumulative Entities", color=COLOR_SECONDARY)
    ax2.tick_params(axis="y", labelcolor=COLOR_SECONDARY)

    ax1.set_title("Platform Growth: Entity Registrations Over Time", fontsize=14, fontweight="bold")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "registration_trend.png", dpi=150, bbox_inches="tight")
    plt.show()

    no_year = len(df_entities) - len(df_year)
    print(f"Note: {no_year:,} entities ({no_year/len(df_entities):.0%}) have no registration year data.")
else:
    print("No entity data available; skipping registration year trend analysis.")

**Key finding:** The platform shows a clear growth trajectory. The year-over-year registration volume indicates increasing adoption, though the pace and any seasonal patterns should be monitored to assess market momentum.

---
## 7. Expansion Prioritization Framework

To identify the highest-value provinces for platform expansion, we build a composite scoring model:

| Factor | Weight | Rationale |
|--------|--------|-----------|
| **Gap Score** | 60% | Larger gap = more unreached entities |
| **Density Score** | 40% | Larger market = higher absolute opportunity |

In [ ]:
# Scoring model for province prioritization
df_priority = df.copy()

# Normalize scores to 0-1 range
gap_min, gap_max = df_priority["coverage_gap"].min(), df_priority["coverage_gap"].max()
density_min, density_max = df_priority["entities_total"].min(), df_priority["entities_total"].max()

if gap_max > gap_min:
    df_priority["gap_score"] = (df_priority["coverage_gap"] - gap_min) / (gap_max - gap_min)
else:
    df_priority["gap_score"] = 0.0

if density_max > density_min:
    df_priority["density_score"] = (df_priority["entities_total"] - density_min) / (density_max - density_min)
else:
    df_priority["density_score"] = 0.0

df_priority["priority_score"] = 0.6 * df_priority["gap_score"] + 0.4 * df_priority["density_score"]

# Assign tiers by quartile
df_priority["priority_tier"] = pd.qcut(
    df_priority["priority_score"].rank(method="first"),
    q=4, labels=["Tier 4", "Tier 3", "Tier 2", "Tier 1"]
)

df_priority = df_priority.sort_values("priority_score", ascending=False)

print("Priority tier distribution:")
print(df_priority["priority_tier"].value_counts().sort_index())

In [ ]:
# Priority Matrix: Market Size vs Coverage Gap
fig, ax = plt.subplots(figsize=(12, 9))

tier_colors = {"Tier 1": "#DC2626", "Tier 2": "#F59E0B", "Tier 3": "#3B82F6", "Tier 4": "#9CA3AF"}

for tier, color in tier_colors.items():
    mask = df_priority["priority_tier"] == tier
    ax.scatter(
        df_priority.loc[mask, "entities_total"],
        df_priority.loc[mask, "coverage_gap"],
        c=color, label=tier, s=100, edgecolors="white", linewidth=0.8, zorder=5,
    )

# Quadrant lines at median
med_x = df_priority["entities_total"].median()
med_y = df_priority["coverage_gap"].median()
ax.axvline(med_x, color="gray", linestyle=":", alpha=0.5)
ax.axhline(med_y, color="gray", linestyle=":", alpha=0.5)

# Annotate top 10 by priority
for _, row in df_priority.head(10).iterrows():
    ax.annotate(
        f"{row['province_name']} ({row['province_abbr']})",
        (row["entities_total"], row["coverage_gap"]),
        textcoords="offset points", xytext=(8, 6), fontsize=8,
        arrowprops={"arrowstyle": "->", "color":"gray", "lw":0.7},
    )

ax.set_xlabel("Market Size (Registry Entities)", fontsize=12)
ax.set_ylabel("Coverage Gap (Unreached Entities)", fontsize=12)
ax.set_title("Expansion Priority Matrix", fontsize=14, fontweight="bold")
ax.legend(title="Priority Tier", fontsize=10)

# Quadrant labels
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.text(xlim[1] * 0.75, ylim[1] * 0.92, "HIGH PRIORITY", fontsize=11,
        color="#DC2626", fontweight="bold", ha="center", alpha=0.6)
ax.text(xlim[0] + (med_x - xlim[0]) * 0.5, ylim[0] + (med_y - ylim[0]) * 0.3,
        "LOW PRIORITY", fontsize=11, color="gray", fontweight="bold", ha="center", alpha=0.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "priority_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top 20 provinces for expansion
top20_cols = [
    "region_name", "province_name", "province_abbr",
    "entities_total", "platform_entities", "coverage_gap",
    "coverage_ratio", "priority_score", "priority_tier",
]
print("Top 20 Provinces for Expansion:")
display(
    df_priority.head(20)[top20_cols]
    .reset_index(drop=True)
    .style.format({
        "coverage_ratio": "{:.1%}",
        "priority_score": "{:.3f}",
        "entities_total": "{:,}",
        "platform_entities": "{:,}",
        "coverage_gap": "{:,}",
    })
    .bar(subset=["priority_score"], color=COLOR_GAP, vmin=0, vmax=1)
)

In [ ]:
if df_exploded.empty:
    print("No sport data available; skipping sport opportunity index.")
else:
    # Sport Opportunity by Region: how many of the top-10 macro-categories
    # have fewer than 10 entities in each region (= underserved)
    top10_sports = df_exploded["sport_macro"].value_counts().head(10).index.tolist()

    sport_region_counts = (
        df_exploded[df_exploded["sport_macro"].isin(top10_sports)]
        .groupby(["region_name", "sport_macro"])
        .size()
        .unstack(fill_value=0)
    )

    # Count sports with < 10 entities per region (= opportunity)
    sport_gaps = (sport_region_counts < 10).sum(axis=1).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(sport_gaps.index, sport_gaps.values, color=COLOR_SECONDARY)

    for bar, val in zip(bars, sport_gaps.values):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                f"{val}", va="center", fontsize=9)

    ax.set_xlabel(f"Number of Top-10 Sports with < 10 Entities (out of {len(top10_sports)})")
    ax.set_title("Sport Opportunity Index by Region", fontsize=14, fontweight="bold")
    ax.set_xlim(0, len(top10_sports) + 1)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "sport_opportunity_by_region.png", dpi=150, bbox_inches="tight")
    plt.show()

### Recommended Expansion Strategy

Based on the priority framework, a three-phase approach:

| Phase | Focus | Criteria |
|-------|-------|---------|
| **Phase 1** | Geographic expansion | Tier-1 provinces — largest market + largest gap |
| **Phase 2** | Sport deepening | Add under-represented sport categories in existing regions |
| **Phase 3** | Long-tail expansion | Tier-2 provinces — medium market, moderate gap |

---
## 8. Geographic Visualization

In [ ]:
# Choropleth map of Italy: platform entities by province
try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False
    print("geopandas not installed. Install it to generate the choropleth map:")
    print("  pip install geopandas")

if HAS_GEOPANDAS:
    geo_path = PROJECT_ROOT / "data" / "geo" / "provinces.geojson"

    if not geo_path.exists():
        print(f"GeoJSON not found at {geo_path}")
        print("Copy the geographic boundaries from data_sample/geo/ to data/geo/:")
        print("  mkdir -p data && cp -r data_sample/geo data/geo")
        HAS_GEOPANDAS = False

if HAS_GEOPANDAS:
    import matplotlib.patheffects as pe
    from matplotlib.colors import LinearSegmentedColormap, PowerNorm

    gdf = gpd.read_file(geo_path)

    acr_col = None
    for candidate in ["prov_acr", "sigla", "SIGLA", "prov_sigla", "COD_PROV"]:
        if candidate in gdf.columns:
            acr_col = candidate
            break

    if acr_col:
        gdf = gdf.merge(df_platform[["province_abbr", "platform_entities"]],
                        left_on=acr_col, right_on="province_abbr", how="left")
        gdf["platform_entities"] = gdf["platform_entities"].fillna(0)

        # Custom colormap: red (low) -> yellow -> green (high)
        cmap_go = LinearSegmentedColormap.from_list(
            "red_green", ["#DC2626", "#FACC15", "#16A34A"]
        )

        # Power normalization: spreads low values across more colors (gamma < 1)
        vmin = gdf["platform_entities"].min()
        vmax = gdf["platform_entities"].max()
        norm = PowerNorm(gamma=0.4, vmin=vmin, vmax=vmax)

        fig, ax = plt.subplots(1, 1, figsize=(14, 16))
        gdf.plot(
            column="platform_entities", ax=ax, legend=True,
            cmap=cmap_go, norm=norm, edgecolor="gray", linewidth=0.3,
            legend_kwds={"label": "Platform Entities", "shrink": 0.6},
            missing_kwds={"color": "lightgray", "label": "No data"},
        )

        # Label each province with its abbreviation at the representative point
        for _, row in gdf.iterrows():
            point = row.geometry.representative_point()
            label = row.get(acr_col, "")
            if label:
                ax.annotate(
                    label,
                    xy=(point.x, point.y),
                    ha="center", va="center",
                    fontsize=8, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=2, foreground="black")],
                )

        ax.set_title("Platform Coverage Across Italian Provinces",
                     fontsize=16, fontweight="bold")
        ax.axis("off")

        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "italy_choropleth.png", dpi=150, bbox_inches="tight")
        plt.show()

        print(f"Entities range: {vmin:.0f} – {vmax:.0f}")
        print(f"Median: {gdf['platform_entities'].median():.0f}")
    else:
        print(f"Could not find province abbreviation column. Available: {list(gdf.columns)}")

---
## 9. Conclusions & Recommendations

### Executive Summary

1. **Market sizing:** In the current dataset, the platform is present across all provinces covered in this snapshot (107 in the current extract), but coverage depth varies enormously — a handful of provinces concentrate most of the activity.

2. **Coverage gap:** Where registry data is available, the gap between total registered entities and platform presence is substantial. The platform has captured only a fraction of the total addressable market.

3. **Sport concentration:** Football dominates the platform's sport mix. The top 3 macro-categories account for over half of all sport-entity pairs, creating both a strength (clear beachhead) and a risk (limited diversification).

4. **Growth trajectory:** Registration data shows sustained growth over recent years, indicating positive market momentum.

5. **Expansion opportunity:** The priority framework identifies high-value provinces where large markets intersect with large gaps — these represent the most impactful targets for geographic expansion.

### Next Steps

- Run the full registry pipeline (`FETCH_REGISTRY_DATA=true`) for complete coverage gap analysis across all provinces covered in the registry snapshot (107 in the current extract).
- Enrich with demographic data (population, income) for more robust prioritization.
- Monitor registration trends monthly to detect acceleration or slowdown.
- Explore the interactive dashboard on [Looker Studio](https://lookerstudio.google.com/s/tDAIpFPxjls) for drill-down exploration.

In [ ]:
# Export analysis outputs
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Coverage gap by province (merged)
output_gap = ANALYSIS_DIR / "coverage_gap_by_province.csv"
df.to_csv(output_gap, index=False)
print(f"Saved: {output_gap}")

# Expansion priority ranking
output_priority = ANALYSIS_DIR / "expansion_priority_by_province.csv"
df_priority[top20_cols].to_csv(output_priority, index=False)
print(f"Saved: {output_priority}")

# Sport-level aggregation by region
output_sport = ANALYSIS_DIR / "platform_sport_by_region.csv"
(
    df_exploded
    .groupby(["region_name", "sport_macro"])
    .size()
    .reset_index(name="sport_entity_pairs")
    .to_csv(output_sport, index=False)
)
print(f"Saved: {output_sport}")

print(f"\nAll outputs saved to {ANALYSIS_DIR}")